#### 1. Librerías.

In [1]:
%run "./librerias/librerias.ipynb"

#### 2. Constantes.

In [2]:
#a. Modo de ejecución (se define en ./constantes/modo.txt, un solo lugar para los 4 notebooks).
# "validacion" = entreno solo con train, puedo medir nDCG.
# "entrega"    = entreno con train+test, uso todo el historial para predecir.
with open("./constantes/modo.txt") as f:
    MODO = f.read().strip()

assert MODO in ("validacion", "entrega"), f"MODO inválido: {MODO!r}"
print(f"MODO: {MODO}")

MODO: validacion


In [3]:
#b. Otras constantes.
%run "./constantes/constantes.ipynb"

In [4]:
#c. Verificación del modo (que los paths coincidan con lo que creo que estoy corriendo).
print(f"MODO: {MODO} | sufijo: {sufijo!r}")
print(f"train_fe: {path_train_fe}")
print(f"modelo:   {path_modelo}")

MODO: validacion | sufijo: ''
train_fe: ./inputs/train_fe.csv
modelo:   ./modelos/modelo.pkl


#### 3. Funciones.

In [5]:
%run "./funciones/funciones.ipynb"

#### 4. Lecturas.

In [6]:
#a. Train.
df_train = pd.read_csv(path_train_fe)

In [7]:
#b. Test (solo hace falta para evaluar; en entrega ya está dentro de la base).
if MODO == "validacion":
    df_test = pd.read_csv(path_test_crudo, dtype={"id_lector": str, "id_libro": str})

In [8]:
#c. Dataset a predecir.
df_a_predecir = pd.read_csv(path_a_predecir)

In [9]:
#d. Libros y Lectores (lo tomo para armar df_test).
df_libros = pd.read_csv(path_libros_fe)
df_lectores = pd.read_csv(path_lectores_fe)

#### 5. Preparación previa.

In [10]:
#a. Forma final.
print(f"Train: {df_train.shape}")
print("\n")

Train: (383547, 122)




In [11]:
#b. Me aseguro que no hayan quedado nulos en los ratings.
cols_rating = [c for c in df_train.columns if c.startswith("rating_prom")]
print("Train.")
print(df_train[cols_rating].isna().sum())

Train.
rating_prom_id_lector                          0
rating_prom_id_lector_autor                    0
rating_prom_id_lector_genero_libro_agrupado    0
rating_prom_id_libro                           0
rating_prom_autor                              0
rating_prom_genero                             0
dtype: int64


In [12]:
#c. Armo X e y. 
features_base = [
    "anio_edicion", 
    "nacimiento",
    #"edad_al_interactuar", 
    #"dias_transcurridos_interaccion", # Es 0 en test/producción siempre.
    #"anios_transcurridos_edicion",
    #"antiguedad_libro_hoy", # Es igual a anios_transcurridos_edicion en test/producción.
    'frecuencia_lector', 
    'frecuencia_libro', 
    'n_interacciones_lector_autor',
    'n_interacciones_lector_genero', 
    'n_lectores_distintos_autor',
    #'rating_prom_id_lector', 
    'rating_prom_id_lector_autor',
    'rating_prom_id_lector_genero_libro_agrupado', 
    #'rating_prom_id_libro', # Con RF sobre RMSE, las features de calidad de ítem degradan el ranking porque desplazan a las de afinidad. Usar con un algoritmo que ordene, no que sea pointwise. 
    #'rating_prom_autor',    # Con RF sobre RMSE, las features de calidad de ítem degradan el ranking porque desplazan a las de afinidad. Usar con un algoritmo que ordene, no que sea pointwise. 
    #'rating_prom_genero',   # Con RF sobre RMSE, las features de calidad de ítem degradan el ranking porque desplazan a las de afinidad. Usar con un algoritmo que ordene, no que sea pointwise. 
    'n_autores_distintos_lector',
    'n_generos_distintos_lector'
]
features_dummies = [c for c in df_train.columns if c.startswith((
    "genero_persona_", 
    #"genero_libro_agrupado_", 
    #"editorial_agrupada_", 
    #"pais_agrupado_"
))]
features = features_base + features_dummies

X = df_train[features]
y = df_train["rating"]

In [13]:
#d. Entreno con todas las filas en ambos modos: el nDCG no usa el holdout interno
# (se calcula sobre df_test con el loop de retrieval), así que separar un 20%
# solo sirve para el RMSE y hace que el modelo que valido no sea el que entrego.
X_train, y_train = X, y
print(f"Modo {MODO}: entreno con las {len(X)} filas.")

Modo validacion: entreno con las 383547 filas.


#### 6. Entrenamiento.

In [ ]:
#a. Busco hiperparametros con RandomizedSearchCV.
#i. Espacio de búsqueda.
#param_dist = {
#    "n_estimators": [100, 200, 500],
#    "max_depth": [10, 20, 30],
#    "min_samples_leaf": [3, 5, 10],
#    "min_samples_split": [5, 10],
#    "max_features": ["sqrt", "log2"],
#}

#ii. Armo la búsqueda.
#busqueda = RandomizedSearchCV(
#    estimator=RandomForestRegressor(random_state=42, n_jobs=1),
#    param_distributions=param_dist,
#    n_iter=30,
#    scoring="neg_root_mean_squared_error",
#    cv=3,
#    random_state=42,
#    verbose=2,
#    n_jobs=-1,
#)

#iii. Corro la búsqueda sobre train.
#busqueda.fit(X_train, y_train)

#iv. Reviso los mejores hiperparámetros encontrados.
#print("Mejores hiperparámetros:", busqueda.best_params_)
#print("Mejor RMSE (CV, promedio de los 3 folds):", -busqueda.best_score_)

# Si quiero dejar congelado los hiperparámetros.
rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=30,
    min_samples_leaf=3,
    min_samples_split=10,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train,y_train)

RandomForestRegressor(max_depth=30, max_features='sqrt', min_samples_leaf=3,
                      min_samples_split=10, n_estimators=500, n_jobs=-1,
                      random_state=42)

In [ ]:
#b. Exportamos el modelo entrenado.
joblib.dump(rf, path_modelo)
print(f"Modelo exportado con exito en: {path_modelo}")

Modelo exportado con exito en: ./modelos/modelo.pkl


In [ ]:
#c. En modo entrega el notebook termina aca: lo que sigue es evaluacion,
# y no tiene sentido evaluar contra un test que ya esta dentro del entrenamiento.
if MODO != "validacion":
    print("\nModo entrega: fin del notebook 3. Segui con el notebook 4.")
    raise KeyboardInterrupt("Corte intencional: modo entrega.")

#### 7. Evaluación del modelo sobre test -  nDCG@20.

In [14]:
#a. Precómputos (fuera del loop).
#1. Universo de libros.
conn = sqlite3.connect(path_db)
todos_los_libros = pd.read_sql("SELECT id_libro FROM interacciones", conn)["id_libro"].unique()
conn.close()
#2. Historial por lector.
leidos_por_lector = df_train[["id_lector","id_libro"]].groupby("id_lector")["id_libro"].apply(set).to_dict()
#3. Ground truth por lector.
gt_por_lector = {lid: pd.Series(g["rating"].values, index=g["id_libro"].values)
                 for lid, g in df_test.groupby("id_lector")}
#4. Lectores de test a recomendarle.
id_lectores_test = df_test["id_lector"].unique()

In [15]:
df_libros[(df_libros["id_libro"].isin(todos_los_libros))&(df_libros["isbn"].isna())]

,id_libro,titulo,autor,genero_libro,editorial,anio_edicion,isbn,resumen,img_src,genero_libro_agrupado,editorial_agrupada
40750,quienes-somos,NaN,NaN,NaN,NaN,2011,NaN,NaN,NaN,sin_dato,otras


In [16]:
print(len(set(todos_los_libros) - set(df_libros["id_libro"])))

70


,id_libro,titulo,autor,genero_libro,editorial,anio_edicion,isbn,resumen,img_src,genero_libro_agrupado,editorial_agrupada
40750,quienes-somos,NaN,NaN,NaN,NaN,2011,NaN,NaN,NaN,sin_dato,otras


In [18]:
#b. Tablas de referencia para el feature engineering de los candidatos.
#i. Armo las tablas.
(media_global, caract_lector, caract_libros,
 afinidad_lector_autor_test, afinidad_lector_genero_test) = armar_tablas_referencia(
    df_train, df_libros, df_lectores
)

Nulos caract_lector: ninguno
Nulos caract_libros: {'autor': 78227, 'genero_libro_agrupado': 78220}
Duplicados (lector / libro / lector-autor / lector-genero): 0 0 0 0
Lectores: 11285 | Libros: 128743 | Libros sin interacciones en la base: 84772


In [ ]:
#ii. Comprobación.
print(
    "Duplicados lectores:",
    caract_lector["id_lector"].duplicated().sum()
)

print(
    "Duplicados libros:",
    caract_libros["id_libro"].zduplicated().sum()
)

print(
    "Duplicados lector-autor:",
    afinidad_lector_autor_test
    .duplicated(["id_lector", "autor"])
    .sum()
)

print(
    "Duplicados lector-género:",
    afinidad_lector_genero_test
    .duplicated(["id_lector", "genero_libro_agrupado"])
    .sum()
)

Duplicados lectores: 0
Duplicados libros: 0
Duplicados lector-autor: 0
Duplicados lector-género: 0


In [ ]:
#c. Genero una muestra de test (para no correrlo todos).
#lectores_test_muestra = (z
#    df_test["id_lector"]
#    .drop_duplicates()
#    .sample(
#        n=1000,
#        random_state=42
#    )
#    .tolist()
#)

#c. Genero una muestra de test estratificada por frecuencia, para que la composición
# se parezca a la de los lectores que evalúa Kaggle (mediana ~103, la mía era ~45).
# Sin esto optimizo para lectores livianos y me miden sobre pesados.
freq_train = df_train.groupby("id_lector").size()
bins = [-1, 0, 20, 50, 100, 250, np.inf]

#i. Distribución objetivo: la de los lectores a predecir.
freq_objetivo = df_a_predecir["id_lector"].map(freq_train).fillna(0)
pesos_objetivo = pd.cut(freq_objetivo, bins).value_counts(normalize=True)

#ii. Lectores disponibles en test, con su bucket.
disponibles = pd.DataFrame({"id_lector": df_test["id_lector"].drop_duplicates().sort_values()})
disponibles["freq"] = disponibles["id_lector"].map(freq_train).fillna(0)
disponibles["bucket"] = pd.cut(disponibles["freq"], bins)

#iii. Muestreo cada bucket según el peso objetivo (o todo lo que haya, si no alcanza).
n_total = 1000
partes = []
for bucket, peso in pesos_objetivo.items():
    pool = disponibles[disponibles["bucket"] == bucket]
    n_pedido = int(round(peso * n_total))
    partes.append(pool.sample(n=min(n_pedido, len(pool)), random_state=42))

muestra = pd.concat(partes)
lectores_test_muestra = muestra["id_lector"].tolist()

#iv. Comprobación.
print(f"Muestra: {len(lectores_test_muestra)} lectores")
print(pd.DataFrame({
    "objetivo": pesos_objetivo,
    "muestra": muestra["bucket"].value_counts(normalize=True)
}).round(3))

Muestra: 1001 lectores
                objetivo  muestra
(0.0, 20.0]        0.281    0.281
(100.0, 250.0]     0.236    0.236
(250.0, inf]       0.173    0.173
(50.0, 100.0]      0.159    0.159
(20.0, 50.0]       0.129    0.129
(-1.0, 0.0]        0.023    0.023


In [21]:
#d. Calculamos.
#i. Lista vacía donde iré almacenando los nDCG de cada usuario, y datos para luego monitorear.
ndcg_lista = []
total_lectores = len(lectores_test_muestra)
print("Comienza la predicción general.")
#ii. Recorro cada id_lector a recomendar.
for i, id_lector in enumerate(lectores_test_muestra, start=1):
    print("\nLector: {} {}/{}".format(id_lector, i, total_lectores))
    #1. Me traigo los libros a recomendarle al id_lector.
    #print("1. Retrieval.")
    libros_candidatos_a_recomendar = retrieval(id_lector)

    #2. Me traigo el Ground Truth del lector (que por diseño, son 20 en test).
    #print("2. Me traigo el True Relevance.")
    true_relevance = gt_por_lector[id_lector]

    #3. Realizo el feature engineering sobre todos los libros candidatos a recomendar.
    #print("3. Armo las features de los libros candidatos.")
    df_features_candidatos = feature_engineering_test(id_lector, libros_candidatos_a_recomendar)

    #4. Predigo el ranking para cada libro candidato del id_lector
    #print("4. Predigo sobre los libros candidatos su rating.")
    predicted_scores_dict = ranking(df_features_candidatos, features, rf)

    #5. Creo una lista de los libros verdaderos + los que evalué, universo en común a evaluar.
    id_libros = list(set(true_relevance.index) |set(predicted_scores_dict.keys()))

    #6. Traigo el rating real de cada libro. Si no es relevante, entonces = 0.
    y_true = np.asarray([[true_relevance.get(id_libro, 0) for id_libro in id_libros]])

    #7. Traigo el rating predicho por el modelo de cada libro. Si no lo evaluó, entonces = 0.
    y_score = np.asarray([[predicted_scores_dict.get(id_libro, 0) for id_libro in id_libros]])

    #8. Calculo el nDCG@20 para el id_lector.
    #print("5. Calculo el nDCG para el lector en cuestión.")
    ndcg = ndcg_score(y_true, y_score, k=20)

    #9. Lo agrego a la lista.
    ndcg_lista.append(ndcg)

    #10. Imprimo el nDCG del id_lector.
    print("El nCDG es de:{}".format(ndcg))
    
#iii. Imprimo el nDCG promedio de todos los id_lectores a predecir.
ndcg_arr = np.array(ndcg_lista)
print("\nEl nDCG@20 promedio es: {:.4f} ± {:.4f} (SE)".format(ndcg_arr.mean(), ndcg_arr.std(ddof=1) / np.sqrt(len(ndcg_arr))))

Comienza la predicción general.

Lector: msor 1/1001
El nCDG es de:0.0

Lector: pmarre 2/1001
El nCDG es de:0.22892428671909032

Lector: galu 3/1001
El nCDG es de:0.0

Lector: ishmael 4/1001
El nCDG es de:0.16832638224488078

Lector: mibagoz 5/1001
El nCDG es de:0.0

Lector: andrestorres 6/1001
El nCDG es de:0.14613388727877552

Lector: jose-m-santos 7/1001
El nCDG es de:0.0

Lector: albertocm 8/1001
El nCDG es de:0.0

Lector: sony69 9/1001
El nCDG es de:0.039454432976902085

Lector: rubio-rose 10/1001
El nCDG es de:0.0

Lector: misaeltv1 11/1001
El nCDG es de:0.20701303746650898

Lector: amaiauc 12/1001
El nCDG es de:0.13019370594783017

Lector: lisa210 13/1001
El nCDG es de:0.0

Lector: mmj 14/1001
El nCDG es de:0.09555514044680816

Lector: mihernan 15/1001
El nCDG es de:0.22242261183009296

Lector: qqq25 16/1001
El nCDG es de:0.0

Lector: martinalvarezalcaire 17/1001
El nCDG es de:0.24310415958872386

Lector: niobe29 18/1001
El nCDG es de:0.3126753014978023

Lector: merchesotodeltor

In [25]:
print("Mediana de frecuencia — muestra:", muestra["freq"].median())
print("Mediana de frecuencia — Kaggle:", freq_objetivo.median())

Mediana de frecuencia — muestra: 69.0
Mediana de frecuencia — Kaggle: 72.0


#### 9. Análisis.

In [22]:
#a. Diagnóstico: dónde gana y dónde pierde el modelo.
freq = df_train.groupby("id_lector").size()

res = pd.DataFrame({"id_lector": lectores_test_muestra, "ndcg": ndcg_lista})
res["freq"] = res["id_lector"].map(freq).fillna(0)
res["bucket"] = pd.cut(res["freq"], [-1, 0, 20, 50, 100, 250, np.inf],
                       labels=["cold", "1-20", "21-50", "51-100", "101-250", "250+"])

print(res.groupby("bucket", observed=True)["ndcg"].agg(["mean", "median", "count"]))
print("\nLectores con nDCG = 0:", (res["ndcg"] == 0).mean())

             mean    median  count
bucket                            
cold     0.000000  0.000000     23
1-20     0.102043  0.041572    281
21-50    0.079955  0.030170    129
51-100   0.070246  0.024944    159
101-250  0.033623  0.000000    236
250+     0.026542  0.000000    173

Lectores con nDCG = 0: 0.5594405594405595


In [23]:
#b. Techo de recall: ¿los 20 libros del GT son siquiera candidatos?
recalls = []
for lid in lectores_test_muestra[:200]:
    cands = set(retrieval(lid))
    gt = set(gt_por_lector[lid].index)
    recalls.append(len(cands & gt) / len(gt))
print("Recall del retrieval:", np.mean(recalls))

Recall del retrieval: 1.0


In [24]:
# Comentario.
#Para un lector fijo, el score de un candidato lo deciden rating_prom_id_lector_autor, 
# n_interacciones_lector_autor y las dos de género. 
# Un lector con 15 libros leyó quizá 12 autores: el modelo tiene que elegir entre "autor que 
# ya leyó" (12 candidatos, señal fuerte) y "autor desconocido" 
# (todo el resto, n=0 y rating=media_global). 
# Es una decisión casi binaria y acierta seguido, porque la gente vuelve a sus autores.

#Un lector con 200 libros leyó 150 autores. Ahora hay 150 candidatos con señal positiva y el modelo
#  tiene que ordenarlos entre sí. 
# Eso ya no lo resuelve n_interacciones_lector_autor, que satura. 
# Y las features que podrían desempatar —calidad del libro, popularidad, novedad— son las tres que
#  tenés comentadas.

#O sea: tus features distinguen bien "conocido vs desconocido" y no distinguen nada dentro de 
# "conocido". El lector pesado vive enteramente dentro de esa segunda categoría.

# Y justamente en el dataset a predecir, la mayoría son personas con muchos libros leídos.
# Osea, los estoy subestimando.